In [0]:
# %pip install openpyxl
%restart_python


In [0]:
def excel_to_dict_list(file_path: str, sheet_name: str = None) -> list:
    """
    Reads an Excel file and returns its content as a list of dictionaries.
    :param file_path: Path to the Excel file.
    :param sheet_name: Optional sheet name to read. If None, reads the first sheet.
    :return: List of dictionaries, each representing a row.
    """
    df = pd.read_excel(file_path, sheet_name=sheet_name, dtype=str)
    return df.to_dict(orient='records')

In [0]:
def transform_excel_into_csv(file_path: str, file_name: str) -> None:
    """
    Converts an Excel file to CSV format.
    :param file_path: The path to the directory containing the Excel file.
    :param file_name: The name of the Excel file
    :return None
    """
    try:
        df = pd.read_excel(file_path + file_name, "in", dtype=str)
        df.to_csv(file_path + file_name.replace(".xlsx", ".csv"), index=False)
        print(f"File {file_name} converted to CSV successfully.")
    except Exception as e:
        print(f"Error converting {file_name} to CSV: {e}")

In [0]:
def clean_csv_file(file_path: str, columns: list) -> None:
    """
    Create a new file called _filtered from the original file, removing invalid records and columns.
    :param file_path: The path to the directory containing the CSV file.
    :param columns: List of columns to extract from the original file
    :return None
    """
    try:
        buffer: list = []
        with open(file_path + file_name, "rb") as csv_file:
            headers: bytes = csv_file.readline()
            print(headers)
            for line in csv_file:
                if line.count(",") != len(columns) - 1:
                    continue
                buffer.append(line)

        #with open(file_path + file_name.replace(".csv", "_filtered.csv"), "w")

    except Exception as e:
        print(f"Error filtering {file_name}: {e}")

In [0]:
%sql
-- Step 1: Add cluster_tag column if it doesn't exist
ALTER TABLE dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.Rivendel_matched_2026Q2
ADD COLUMN  cluster_tag INT;

-- Step 2: Populate cluster_tag based on source_file_name
UPDATE dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.Rivendel_matched_2026Q2
SET cluster_tag = CASE source_file_name
    WHEN 'matches_Brokers_Rivendell_70_percent (1).xlsx' THEN 9
    WHEN 'matches_Claims_Rivendell_70_percent.xlsx'      THEN 4
    WHEN 'matches_Cluster3_Rivendell_70_percent.xlsx'    THEN 3
    WHEN 'matches_Cluster7_Rivendell_70_percent.xlsx'    THEN 7
    WHEN 'matches_Cluster8_Rivendell_70_percent.xlsx'    THEN 8
    WHEN 'matches_Cluster9_Rivendell_70_percent.xlsx'    THEN 9
    WHEN 'matches_Financials_Rivendell_70_percent.xlsx'  THEN 5
    ELSE NULL
END;

In [0]:
%sql
select source_file_name, count(distinct bo_documentId
) from dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.Rivendel_matched_2026Q2 RM1
group by source_file_name

In [0]:
%sql
SELECT DISTINCT
    RM1.*,
    CASE WHEN UPPER(TRIM(cms1.created)) = 'ADMINISTRATOR' THEN cms1.lastAuthor ELSE cms1.created END AS created,
    up1.BU
FROM dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.Rivendel_matched_2026Q2 RM1
LEFT JOIN dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_metadata_cms cms1
    ON UPPER(TRIM(RM1.bo_documentId)) = UPPER(TRIM(cms1.Document_Id))
LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.bo_user_profile up1
    ON UPPER(TRIM(up1.UserName)) = UPPER(TRIM(
        CASE WHEN UPPER(TRIM(cms1.created)) = 'ADMINISTRATOR' THEN cms1.lastAuthor ELSE cms1.created END
    ))

In [0]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# --- BU Group mapping ---
BU_GROUPS = {
    'COMMERCIALS': [
        'COMCE-GERMANY,AUSTRIA, SWITZERL,CEE', 'COMAS-ASIA', 'GLOBA-GLOBAL',
        'COMNA-NORTH AMERICA', 'COMNN-NETHERLANDS', 'COMON-OCEANIA',
        'COMSE-BELGIUM, LUXEMBOURG', 'COMSPB-SPAIN, PORTUGAL, BRAZIL',
        'COMUI-UK & IRELAND', 'FRANC-FRANCE', 'ITALY-ITALY', 'CREDIT INSURANCE'
    ],
    'RISK': [
        'RISK1-RS1-GERMANY, CENTRAL & EAST EUROPE', 'RISK2-RS2-BELGIUM, LUX, FRANCE & ITA',
        'RISK3-RS3-NLD, APAC & AIM', 'RISK4-RS4-NORTH EUROPE, CIS & ACS',
        'RISK4-RS4-NORTH EUROPE, CIS & SP', 'RISK5-RS5-NAFTA',
        'RISK1-RS1-DEU, AUT and CHE', 'RISK1-RS1-Europe, Russia & CIS',
        'RISK2-RS2-NLD, Belux, FRA, Africa & ISR', 'RISK3-RS3-Asia and Oceania',
        'RISK7-RS7-Spain, Portugal, Andorra'
    ],
    'FINANCE': [
        'FINPM-FINANCE PROGRAM MANAGEMENT', 'GFO-GROUP FINANCE OPERATIONS',
        'GFC-GROUP FINANCE AND CONTROL', 'COFIN-CORPORATE FINANCE & TAX',
        'Group Finance', 'Finance', 'Finance & Control'
    ],
}

bu_to_group = {bu.upper().strip(): group for group, bus in BU_GROUPS.items() for bu in bus}

def map_bu_group(bu):
    if pd.isna(bu) or str(bu).strip().upper() in ('', 'UNIDENTIFIED', 'LEFT ORGANIZATION'):
        return 'Unidentified'
    return bu_to_group.get(str(bu).upper().strip(), str(bu).strip())

# --- Query 1: Rivendell matched docs with BU (accessor) ---
pdf_chart = spark.sql("""
    SELECT DISTINCT
        RM1.bo_documentId,
        RM1.cluster_tag,
        RM1.source_file_name,
        CASE WHEN UPPER(TRIM(up1.BU)) IN ('LEFT ORGANIZATION', 'UNIDENTIFIED') OR up1.BU IS NULL OR TRIM(up1.BU) = ''
             THEN 'Unidentified'
             ELSE up1.BU END AS BU
    FROM dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.Rivendel_matched_2026Q2 RM1
    LEFT JOIN dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_metadata_cms cms1
        ON UPPER(TRIM(RM1.bo_documentId)) = UPPER(TRIM(cms1.Document_Id))
    LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.bo_user_profile up1
        ON UPPER(TRIM(up1.UserName)) = UPPER(TRIM(
            CASE WHEN UPPER(TRIM(cms1.created)) = 'ADMINISTRATOR' THEN cms1.lastAuthor ELSE cms1.created END
        ))
""").toPandas()

pdf_chart['BU_Group'] = pdf_chart['BU'].apply(map_bu_group)
pdf_chart['source_label'] = (
    pdf_chart['source_file_name']
    .str.replace(r'matches_', '', regex=True)
    .str.replace(r'_Rivendell_70_percent.*\.xlsx$', '', regex=True)
    .str.replace(r' \(\d+\)', '', regex=True)
    .str.strip()
)

# --- Query 2: cluster_owner_analysis with OwnerBU ---
pdf_owner = spark.sql("""
    SELECT DISTINCT
        co.Doc_ID  AS bo_documentId,
        co.cluster AS cluster_tag,
        CASE WHEN UPPER(TRIM(co.OwnerBU)) IN ('LEFT ORGANIZATION', 'UNIDENTIFIED') OR co.OwnerBU IS NULL OR TRIM(co.OwnerBU) = ''
             THEN 'Unidentified'
             ELSE co.OwnerBU END AS OwnerBU
    FROM dataplatform01_central_dev_catalog_01.custom_sap_bo.cluster_owner_analysis co
    INNER JOIN dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.Rivendel_matched_2026Q2 RM1
        ON UPPER(TRIM(co.Doc_ID)) = UPPER(TRIM(RM1.bo_documentId))
""").toPandas()

pdf_owner['OwnerBU_Group'] = pdf_owner['OwnerBU'].apply(map_bu_group)

# --- Helper: build pivot ---
def build_pivot(df, group_col, count_col='bo_documentId'):
    agg = (
        df.groupby(['cluster_tag', group_col])[count_col]
        .nunique().reset_index().rename(columns={count_col: 'cnt'})
    )
    pivot = agg.pivot(index='cluster_tag', columns=group_col, values='cnt').fillna(0).sort_index()
    return agg, pivot

# --- Helper: stacked bar chart ---
def stacked_barh(ax, pivot, cols_sorted, color_map, title, legend_title):
    x = np.arange(len(pivot.index))
    width = 0.55
    bottom = np.zeros(len(pivot))
    for col in cols_sorted:
        vals = pivot[col].values
        ax.bar(x, vals, width=width, bottom=bottom,
               color=color_map[col], label=col, edgecolor='white', linewidth=0.4)
        bottom += vals
    for i, total in enumerate(bottom):
        if total > 0:
            ax.text(x[i], total + bottom.max() * 0.01, f'{int(total):,}',
                    ha='center', va='bottom', fontsize=9, fontweight='bold', color='#333333')
    ax.set_xticks(x)
    ax.set_xticklabels([str(int(c)) for c in pivot.index], fontsize=10)
    ax.set_xlabel('Cluster Tag', fontsize=10)
    ax.set_ylabel('Distinct Documents', fontsize=10)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_ylim(0, bottom.max() * 1.18)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.legend(title=legend_title, bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=7, title_fontsize=8)

# Shared colours
base_colors = plt.cm.tab20.colors
defined_groups = ['COMMERCIALS', 'RISK', 'FINANCE']
group_colors = {'COMMERCIALS': '#336699', 'RISK': '#E07B39', 'FINANCE': '#4CAF50', 'Unidentified': '#BEBEBE'}

def build_cols_and_colors(pivot, extra_colors=None):
    other = sorted([c for c in pivot.columns if c not in defined_groups and c != 'Unidentified'])
    cols = [c for c in defined_groups if c in pivot.columns] + other + \
           (['Unidentified'] if 'Unidentified' in pivot.columns else [])
    cmap = dict(group_colors)
    ec = extra_colors or {}
    idx = 0
    for c in cols:
        if c not in cmap:
            cmap[c] = ec.get(c, base_colors[idx % len(base_colors)])
            idx += 1
    return cols, cmap

# Chart 1: by BU Group (accessor)
_, pivot1 = build_pivot(pdf_chart, 'BU_Group')
cols1, cmap1 = build_cols_and_colors(pivot1)

# Chart 2: by source_file_name
src_colors = {c: base_colors[i % len(base_colors)] for i, c in enumerate(sorted(pdf_chart['source_label'].unique()))}
_, pivot2 = build_pivot(pdf_chart, 'source_label')
cols2 = sorted(pivot2.columns.tolist())
pivot2 = pivot2[cols2]

# Chart 3: by OwnerBU Group (from cluster_owner_analysis)
_, pivot3 = build_pivot(pdf_owner, 'OwnerBU_Group')
cols3, cmap3 = build_cols_and_colors(pivot3)

fig, axes = plt.subplots(1, 3, figsize=(30, 6))

stacked_barh(axes[0], pivot1, cols1, cmap1,
             'Rivendell Docs by Cluster & BU Group\n(Accessor)', 'BU Group')
stacked_barh(axes[1], pivot2, cols2, src_colors,
             'Rivendell Docs by Cluster & Source File', 'Source File')
stacked_barh(axes[2], pivot3, cols3, cmap3,
             'Rivendell Docs by Cluster & Owner BU Group\n(cluster_owner_analysis)', 'Owner BU Group')

plt.suptitle('Rivendell Matched Documents — Cluster Tag Breakdown', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f"Total distinct documents (Rivendell): {pdf_chart['bo_documentId'].nunique():,}")
print(f"Total distinct documents (Owner analysis): {pdf_owner['bo_documentId'].nunique():,}")

In [0]:
%sql
select * from dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.Rivendel_matched_2026Q2